# 2. Control no lineal de un robot aéreo en 2D

Se trabajará con el modelo no lineal de un robot aéreo (drone) en 2 dimensiones. Se controlará de manera directa el empuje ("thrust") colectivo del robot a través de $u_1$ y el momento alrededor del eje $x$ a través de $u_2$.

El estado de este robot se puede describir mediante el siguiente vector:

$$X = [y, z, \phi, \dot{y}, \dot{z},\dot{\phi}]$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Robot2D import RobotAereo2D
from Robot2D import animate_robot_from_data

## Controlador No Lineal

Las ecuaciones no lineales de movimiento de un robot aéreo en 2D son las siguientes:

$$
\ddot{z} = g - \frac{u_1}{m}\cos\phi
\qquad \qquad
\ddot{y} = \frac{u_1}{m}\sin\phi
\qquad \qquad
\ddot{\phi} = \frac{u_2}{I_x}$$

Se utilizará linealización por realimentación, de tal modo que las entradas virtuales serán $v_y$, $v_z$, $v_\phi$. En función de estas entradas virtuales, las entradas son:

$$
u_1 = \frac{m(g - v_z)}{\cos\phi}
\qquad \qquad
\phi_{c} = \sin^{-1}\left(\frac{m v_y}{u_1}\right)
\qquad \qquad
u_2 = I_x v_\phi
$$

donde $v_y$, $v_z$, $v_\phi$ se implementan utilizando leyes PD con un término feedforward de aceleración.

In [ ]:
class ControladorNoLineal:
    """
    Implementa un controlador no lineal en cascada para el robot en 2D
    """
    def __init__(self, m, Ix, kp_z=1.0, kd_z=1.0,
                 kp_y=1.0, kd_y=1.0, kp_phi=1.0, kd_phi=1.0):
        # Ganancias
        self.kp_z = kp_z; self.kd_z = kd_z
        self.kp_y = kp_y; self.kd_y = kd_y
        self.kp_phi = kp_phi; self.kd_phi = kd_phi
        # Parámetros del robot (masa, inercia)
        self.g = 9.81
        self.Ix = Ix
        self.m = m

    def control_z(self, z_des, z, dz_des, dz, ddz_des, phi):
        """
        Controlador PD de la altitud. Genera u1 (propulsión)
        """
        # PD + feedforward
        vz = ddz_des - self.kd_z*(dz-dz_des) - self.kp_z*(z-z_des) 
        # Señal u1 no lineal
        u1 = self.m * (self.g-vz) / np.cos(phi)
        return u1

    def control_y(self, y_des, y, dy_des, dy, u1, ddy_des=0.0):
        """
        Controlador PD lateral. Genera el ángulo phi comandado (phi_c)
        """
        # PD + feedforward
        vy = ddy_des - self.kd_y*(dy-dy_des) - self.kp_y*(y-y_des)  
        # Señal phi comandada
        x = self.m * vy / u1
        # Limitar al rango +-1 (para poder aplicar arcoseno)
        x = min(0.99, max(x,-0.99))
        # Señal phi no lineal (también se podría haber utilizado el arcotangente)
        phi_c = np.arcsin(x)

        return phi_c

    def control_phi(self, phi_des, phi, dphi, dphi_des=0.0, ddphi_des=0.0):
        """
        Controlador PD para el giro (phi). Genera momento u2 alrededor del eje x
        """
        # PD + feedforward
        vphi = ddphi_des - self.kd_phi*(dphi-dphi_des) - self.kp_phi*(phi-phi_des) 
        # Señal u2 (momento) linealizada
        u2 = vphi * self.Ix
        return u2

### Generador de una trayectoria

La trayectoria que se utilizará para probar el controlador es la figura 8 descrita por:
$$
\begin{align}
z & = -a_z \sin{\omega_z t} \\
y & = a_y \cos{\omega_y t}
\end{align}
$$
donde $\omega_y = \omega_z / 2$.

In [ ]:
def trayectoria(duracion, omega_z=1.0, az=1.0, ay=1.0, dt=0.01):
    omega_y = omega_z/2
    t=np.linspace(0.0, duracion, int(duracion/dt))

    # Trayectoria deseada para y
    y   =  ay*np.cos(omega_y*t)
    dy  = -ay*omega_y*np.sin(omega_y*t)
    ddy = -ay*omega_y**2*np.cos(omega_y*t)

    # Trayectoria deseada para z
    z   =  -az*np.sin(omega_z*t)
    dz  =  -az*omega_z*np.cos(omega_z*t)
    ddz = az*omega_z**2*np.sin(omega_z*t)

    return (y, dy, ddy), (z, dz, ddz), t

### Trayectoria deseada

In [ ]:
# Parámetros de la trayectoria
tiempo_total = 30.0
# Generación de la trayectoria (z,y)
yy, zz, t = trayectoria(tiempo_total, dt=0.02)
y_des, dy_des, ddy_des = yy
z_des, dz_des, ddz_des = zz

# Tiempo de simulación
dt = t[1]-t[0]

# Forma de la trayectoria deseada
plt.figure(figsize=(5,3))
plt.plot(y_des, z_des)
plt.title("Trayectoria deseada")
plt.xlabel("y [m]"); plt.ylabel("z [m]");

### Seguimiento de trayectoria

In [ ]:
# Ganancias del controlador
kp_z   = 0.1
kd_z   = 10.0
kp_y   = 0.3
kd_y   = 10.0
kp_phi = 150.0
kd_phi = 50.0

# Robot simulado
robot = RobotAereo2D()

# Estado inicial del robot
X0 = np.array([y_des[0], z_des[0], 0.1,
              dy_des[0], dz_des[0], 0])
#X0 = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
robot.set_state(X0)

# Parámetros estimados del robot
m = 0.2; Ix = 0.1
# Controlador
nlin_controller = ControladorNoLineal(m, Ix, kp_z, kd_z,
                                      kp_y, kd_y, kp_phi, kd_phi)

**Simulación del movimiento**

In [ ]:
# Establecer el estado inicial del robot
robot.set_state(X0)

# Factor para controlar más rápido el lazo interno
factor_lazo_interno = 10

# Almacenamiento del historial de los estados y control
Xt = np.zeros((t.size, 6))
Xt[0,:] = robot.X
Ut = np.zeros((factor_lazo_interno*(t.size-1), 2))

# Ejecución del vuelo
for i in range(0, t.size-1):

    # Estado actual del robot
    y    = robot.X[0];  dy   = robot.X[3]
    z    = robot.X[1];  dz   = robot.X[4]
    phi  = robot.X[2];  dphi = robot.X[5]

    # Controlador
    u1 = nlin_controller.control_z(z_des[i], z, dz_des[i], dz, ddz_des[i], phi)
    phi_c = nlin_controller.control_y(y_des[i], y, dy_des[i], dy, u1, ddy_des[i])

    # Bucle interno (cascada) para controlar phi
    for j in range(factor_lazo_interno):
        # Controlador para u2
        u2 = nlin_controller.control_phi(phi_c, phi, dphi, 0.0 )

        # Simulación del movimiento del robot
        robot.set_control(u1, u2)
        robot_state = robot.advance_state(dt/factor_lazo_interno)

        # Historial de la señal de control
        Ut[i*factor_lazo_interno+j,0] = u1
        Ut[i*factor_lazo_interno+j,1] = u2

    # Generación del historial de las posiciones verticales
    Xt[i+1,:] = robot_state

# Recuperación del estado y control
traj_y = Xt[:,0]; traj_z = Xt[:,1]; traj_phi = Xt[:,2]
traj_dy = Xt[:,3]; traj_dz = Xt[:,4]; traj_dphi = Xt[:,5]
traj_u1 = Ut[:,0]; traj_u2 = Ut[:,1]

### Animación del movimiento del robot

In [ ]:
# Mostrar una animación del vuelo del robot
ts = 10              # Factor de muestreo
porcentaje = 0.4     # Porcentaje (0 a 1) del tiempo total

tf = int(np.floor(porcentaje*(tiempo_total/dt)))
animate_robot_from_data(traj_y[0:tf:ts], traj_z[0:tf:ts], traj_phi[0:tf:ts],
                        y_des[0:tf:ts], z_des[0:tf:ts],
                        dt=dt, title="Animación del Robot 2D")

### Trayectoria



In [ ]:
plt.figure(figsize=(6,4))

plt.plot(y_des, z_des, linestyle='-', marker='.', color='red')
plt.plot(traj_y, traj_z, linestyle='-', color='blue')
plt.title('Trayectoria de vuelo').set_fontsize(10)
plt.xlabel('$y$ [$m$]').set_fontsize(10); plt.ylabel('$z$ [$m$]').set_fontsize(10)
plt.xticks(fontsize=10); plt.yticks(fontsize=10)
plt.gca().invert_yaxis()
plt.legend(['Trayectoria deseada','Trayectoria ejecutada'],fontsize = 10)
plt.axis('equal'); plt.grid()
plt.show()

### Visualización del desplazamiento horizontal y lateral (y sus velocidades)

In [ ]:
plt.figure(figsize=(12,3))

plt.subplot(1,2,1)
plt.plot(t, traj_z); plt.plot(t,z_des)
plt.title('Trayectoria en z (altura)')
plt.xlabel('$t$ [s]'); plt.ylabel('$z$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 14)

plt.subplot(1,2,2)
plt.plot(t, traj_y); plt.plot(t,y_des)
plt.title('Trayectoria en y (lateral)')
plt.xlabel('$t$ [s]'); plt.ylabel('$y$ [$m$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 14)

plt.figure(figsize=(12,3))
plt.subplot(1,2,1)
plt.plot(t, traj_dz); plt.plot(t,dz_des)
plt.title('Velocidad en z')
plt.gca().invert_yaxis()
plt.xlabel('$t$ [s]'); plt.ylabel('$\\dot z$ [$m/s$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 14)

plt.subplot(1,2,2)
plt.plot(t, traj_dy); plt.plot(t,dy_des)
plt.title('Velocidad en y')
plt.xlabel('$t$ [s]'); plt.ylabel('$\\dot y$ [$m/s$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 14)

plt.show()

### Visualización de la orientación y su velocidad

In [ ]:
plt.figure(figsize=(12,3))

plt.subplot(1,2,1)
plt.plot(t, traj_phi); plt.title('Trayectoria de $\\phi$')
plt.xlabel('$t$ [s]'); plt.ylabel('$\\phi$ [$rad$]'); plt.grid()

plt.subplot(1,2,2)
plt.plot(t, traj_dphi); plt.title('Velocidad de $\\phi$')
plt.xlabel('$t$ [s]'); plt.ylabel('$\\dot\\phi$ [$rad/s$]'); plt.grid()

plt.show()

### Visualización de las señales de control

In [ ]:
plt.figure(figsize=(12,3))
t_u = np.linspace(t[0], t[-1], len(traj_u1))

plt.subplot(1,2,1)
plt.plot(t_u, traj_u1); plt.title('Entrada de control $u_1$')
plt.xlabel('$t$ [s]'); plt.ylabel('$u_1$ [N]'); plt.grid()

plt.subplot(1,2,2)
plt.plot(t_u, traj_u2); plt.title('Entrada de control $u_2$')
plt.xlabel('$t$ [s]'); plt.ylabel('$u_2$ [Nm]'); plt.grid()

plt.show()